# **Data processing for MoE Neural Networks**
---

This Notebook generates a dataset in order to be used by an MoE Neural Network.<br><br>
The output dataset combines 2 data sources:

- **Source #01**: `data_engineering_belgique.csv`
- **Source #02**: `predictions_experts.csv` and

**Source #01**: this is an hourly time-series dataset engineered for wind power forecasting, combining the target variable (`Eolien_MW`) with transformed physical and meteorological features.<br>

This dataset spans as follows:
```console
Start date: 2018-01-02 00:00:00
Stop date: 2025-11-23 23:00:00
Samples : 68877 
```

**Source #02**: this dataset contains synthetic experts predictions based on Source #01.

This dataset spans as follows:

```console
Start date: 2024-10-03 00:00:00
Stop date: 2025-11-23 23:00:00
Samples : 10008
```

The generated dataset `experts_and_features_for_moe_nn.csv` will contain the target variable `Eolien_MW`, predictions for three experts : ridge, random forest and LGBM, plus representative representative features from Source #01.<br><br>

Once properly synchronized and merged, this dataset will spans as follows:

```console
Start date: 2024-10-03 00:00:00
Stop date: 2025-11-23 23:00:00
Samples : 10008
```

As you can notice, the span of this dataset is limited by the span of Source #02.

Assuming you have access to Source #01 and Source #02 datastets, the code below allows you to to create this dataset.

In [1]:
# --- Main imports ---
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# --- Project path ---
PROJECT_PATH = os.path.dirname(os.getcwd())
PROJECT_PATH = os.getcwd()

SOURCE_01 = f"{PROJECT_PATH}/data/data_engineering_belgique.csv"
SOURCE_02 = f"{PROJECT_PATH}/data/predictions_experts.csv"

# --- Check files existence ---
files = [SOURCE_01,
         SOURCE_02]

for file in files:
    if os.path.exists(file):
        print(f"File O.k.! :\t{file}")
    else:
        print(f"File NOT O.k.! : {file}")

File O.k.! :	/home/mambo/Documents/Projet_fil_rouge/Neural_Network_Forecasting/data/data_engineering_belgique.csv
File O.k.! :	/home/mambo/Documents/Projet_fil_rouge/Neural_Network_Forecasting/data/predictions_experts.csv


## **Source #01**: `data_engineering_belgique.csv`

In [2]:
df_feat = pd.read_csv(SOURCE_01)

# --- Converts date to time format ---
df_feat["Date_Heure"] = pd.to_datetime(df_feat["Date_Heure"], errors="coerce")

# --- Print span of the dataset ---
print(f"Start date: {df_feat['Date_Heure'].min()}")
print(f"Stop date: {df_feat['Date_Heure'].max()}")
print(f"Samples : {len(df_feat['Date_Heure'])}")

# --- Dataset overview ---
#df_feat.info()
#df_feat.head()

Start date: 2018-01-02 00:00:00
Stop date: 2025-11-23 23:00:00
Samples : 68877


## **Source #02**: `predictions_experts.csv`

In [3]:
df_experts = pd.read_csv(SOURCE_02, sep=";")

# --- Converts date to time format ---
df_experts['Date_Heure'] = pd.to_datetime(df_experts['Date_Heure'], errors="coerce")

# --- Keep only specific columns ---
cols_to_keep = ['Date_Heure', 'y_true', 'Ridge_Global', 'RandomForest_Global', 'LGBM_Global']
df_experts_clean = df_experts[cols_to_keep]

# --- Create a dictionary in order to map old names to new names ---
new_names = {
    "y_true": "y_true",
    "Ridge_Global": "ridge",
    "RandomForest_Global": "randomforest",
    "LGBM_Global": "lgbm"
}

# --- Apply renaming ---
df_experts_clean.rename(columns=new_names, inplace=True)

# --- Print span of the dataset ---
print(f"Start date: {df_experts['Date_Heure'].min()}")
print(f"Stop date: {df_experts['Date_Heure'].max()}")
print(f"Samples : {len(df_experts['Date_Heure'])}")

# --- Dataset overview ---
#df_experts_clean.info()
df_experts_clean.head()

Start date: 2024-10-03 00:00:00
Stop date: 2025-11-23 23:00:00
Samples : 10008


,Date_Heure,y_true,ridge,randomforest,lgbm
0,2024-10-03 00:00:00,1237.84,1249.726980,1164.385482,1045.849256
1,2024-10-03 01:00:00,1105.38,1068.422459,902.434285,790.083609
2,2024-10-03 02:00:00,1097.06,1013.409821,766.710598,714.894714
3,2024-10-03 03:00:00,970.68,918.601422,699.701733,713.470081
4,2024-10-03 04:00:00,1188.70,790.645030,627.655777,585.931413


## **Dataset merge**

In [4]:
# --- Pick representative features ---
features = [
    "Date_Heure",
    "Wind_Norm",
    "Wind_Norm_Cubes",
    "wind_cv_3h",
    "Wind_Dir_Meteo_sin",
    "Wind_Dir_Meteo_cos",
    "Air_density",
    "Hour_sin",
    "Hour_cos",
    "Month_sin",
    "Month_cos"
]

# --- Merge based on "Date_Heure" ---
df_merged = df_experts_clean.merge(
    df_feat[features],
    on="Date_Heure",
    how="left"
)

# --- Save the merged dataset ---
df_merged.to_csv(f"{PROJECT_PATH}/data/experts_and_features_for_moe_nn.csv")

# --- Print span of the dataset ---
print(f"Start date: {df_merged['Date_Heure'].min()}")
print(f"Stop date: {df_merged['Date_Heure'].max()}")
print(f"Samples : {len(df_merged['Date_Heure'])}")

# --- Dataset overview ---
#df_merged.info()
#df_merged.head()

Start date: 2024-10-03 00:00:00
Stop date: 2025-11-23 23:00:00
Samples : 10008
